In [0]:
from pyspark.sql.functions import *

In [0]:
orders_schema = "order_id long,customer_id long,customer_fname string,customer_lname string,city string,state string,pincode long,line_items array<struct<order_item_id: long,order_item_product_id: long,order_item_quantity: long,order_item_product_price: float,order_item_subtotal: float>>"

In [0]:
orders_df=spark.readStream \
    .format("json") \
    .schema(orders_schema) \
    .option("path","/Volumes/dev_analysis/analysis/streaming_practice/files/") \
    .load()



In [0]:
orders_df.createOrReplaceTempView("orders")

In [0]:
exploded_orders=spark.sql(""" select order_id,customer_id,city,state,pincode,
                          explode(line_items) lines from orders""")

In [0]:
exploded_orders.createOrReplaceTempView("exploded_orders")

In [0]:
flattened_orders = spark.sql("""select order_id, customer_id, city, state, pincode, 
lines.order_item_id as item_id, lines.order_item_product_id as product_id,
lines.order_item_quantity as quantity,lines.order_item_product_price as price,
lines.order_item_subtotal as subtotal from exploded_orders""")

In [0]:
def my_function(flattened_orders, batch_id):

    # Create temporary view from incoming micro-batch
    flattened_orders.createOrReplaceTempView("orders_flattened")

    # Aggregate the current micro-batch
    aggregated_orders = spark.sql("""
        SELECT
            customer_id,
            approx_count_distinct(order_id) AS orders_placed,
            count(item_id) AS products_purchased,
            sum(subtotal) AS amount_spent
        FROM orders_flattened
        GROUP BY customer_id
    """)

    # Create temporary view
    aggregated_orders.createOrReplaceTempView("orders_result")

    # Merge current batch aggregates into target table
    merge_statement = """
        MERGE INTO dev_analysis.analysis.orders_final_result t
        USING orders_result s
        ON t.customer_id = s.customer_id

        WHEN MATCHED THEN
            UPDATE SET
                t.products_purchased = t.products_purchased + s.products_purchased,
                t.orders_placed = t.orders_placed + s.orders_placed,
                t.amount_spent = t.amount_spent + s.amount_spent

        WHEN NOT MATCHED THEN
            INSERT *
    """

    spark.sql(merge_statement)

In [0]:
streaming_query = flattened_orders \
.writeStream \
.format("delta") \
.outputMode("update") \
.trigger(availableNow=True) \
.option("checkpointLocation","/Volumes/dev_analysis/analysis/streaming_practice/checkpoint3/") \
.foreachBatch(my_function) \
.start()

In [0]:
%sql
--drop table dev_analysis.analysis.orders_final_result1;
create table if not exists dev_analysis.analysis.orders_final_result1(customer_id long,orders_placed long,products_purchased long,amount_spent float)

In [0]:
spark.sql("""select * from dev_analysis.analysis.orders_final_result""").show()

In [0]:

spark.table("orders_result103").explain()

In [0]:
streaming_query.lastProgress